## Imports

In [1]:
import sys
from pathlib import Path

root_folder = str(Path.cwd().parent)
if root_folder not in sys.path:
    sys.path.append(root_folder)

import zipfile

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 

from sklearn.base import BaseEstimator, TransformerMixin 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, accuracy_score

from utility.utils import GroupedDataSplitter, WindowTransformer, target_windower

## Data Extration from the raw files

In [2]:
with zipfile.ZipFile('../data/archive.zip', 'r') as f:
    print(f'The archive contains: \n{f.namelist()}')
    f.extractall(path='../data/')

The archive contains: 
['adhdata.csv']


In [3]:
dataset = pd.read_csv('../data/adhdata.csv') #, nrows=1500000

## Variable declaration

In [4]:
group_column = "ID" # used for subject-aware windowing, that is, based on individual subjects
target_column = "Class" 
window_size = 512 # number of rows in each window
train_ratio = 0.8 # fraction of the whole dataset in the training set

## Train-Test split by subjects

In [5]:
splitter = GroupedDataSplitter(
    group_column=group_column,
    train_ratio=train_ratio,
    shuffle=True,
    random_state=42,
)

train_data, test_data = splitter.split(dataset)

In [6]:
print(f'train_set: {train_data.shape}\ntest_set: {test_data.shape}')

train_set: (1675871, 21)
test_set: (490512, 21)


In [7]:
train_data['Class'].value_counts() / train_data.shape[0] * 100

Class
ADHD       52.769097
Control    47.230903
Name: count, dtype: float64

In [8]:
# window-level targets for the train and test set

y_train = target_windower(
    train_data,
    group_column=group_column,
    target_column=target_column,
    window_size=window_size,
)

y_test = target_windower(
    test_data,
    group_column=group_column,
    target_column=target_column,
    window_size=window_size,
)

In [9]:
y_train.value_counts() / y_train.shape[0] * 100

ADHD       52.851829
Control    47.148171
Name: count, dtype: float64

In [10]:
y_train = (y_train == 'ADHD').astype('int')
y_train.value_counts() / y_train.shape[0] * 100

1    52.851829
0    47.148171
Name: count, dtype: float64

In [11]:
y_test.value_counts() / y_test.shape[0] * 100

ADHD       65.997888
Control    34.002112
Name: count, dtype: float64

In [12]:
y_test = (y_test == 'ADHD').astype('int')
y_test.value_counts() / y_test.shape[0] * 100

1    65.997888
0    34.002112
Name: count, dtype: float64

In [13]:
print(f'y_train: {y_train.shape}\ny_test: {y_test.shape}')

y_train: (3226,)
y_test: (947,)


In [14]:
# Feature Matrix for train and test set containing the ID column

X_train = train_data.drop(columns=[target_column])
X_test = test_data.drop(columns=[target_column])

In [15]:
print(f'X_train: {X_train.shape}\nX_test: {X_test.shape}')

X_train: (1675871, 20)
X_test: (490512, 20)


In [16]:
# X_test_copy = X_test.copy()
# X_test_copy_trans = WindowTransformer(window_size=512, group_column="ID").transform(X_test_copy)
# print(f'X_test_copy: {X_test_copy.shape}\nX_test_copy_trans: {X_test_copy_trans.shape}\ny_test: {y_test.shape}')

## Feature Engineering

## Modeling

### Model fitting

In [17]:
# Model pipeline

linear_model = Pipeline([
    ("window", WindowTransformer(window_size=512, group_column="ID")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, n_jobs=-1)),
])

linear_model

,steps,"[('window', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,window_size,512
,group_column,'ID'
,stride,512
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'


In [18]:
linear_model.fit(X_train, y_train)

,steps,"[('window', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,window_size,512
,group_column,'ID'
,stride,512
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'


In [19]:
linear_preds = linear_model.predict(X_test)

In [20]:
ensemble_model = Pipeline([
    ("window", WindowTransformer(window_size=512, group_column="ID")),
    ("scaler", StandardScaler()),
    ("clf", HistGradientBoostingClassifier()),
])

ensemble_model

,steps,"[('window', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,window_size,512
,group_column,'ID'
,stride,512
,copy,True
,with_mean,True
,with_std,True
,loss,'log_loss'


In [21]:
ensemble_model.fit(X_train, y_train)

,steps,"[('window', ...), ('scaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,window_size,512
,group_column,'ID'
,stride,512
,copy,True
,with_mean,True
,with_std,True
,loss,'log_loss'


In [22]:
ensemble_preds = ensemble_model.predict(X_test)

### Model Evaluation

In [23]:
print(f'linear_precision_score: {precision_score(y_test, linear_preds)}')
print(f'linear_recall_score: {recall_score(y_test, linear_preds)}')
print(f'linear_accuracy_score: {accuracy_score(y_test, linear_preds)}')

linear_precision_score: 0.680073126142596
linear_recall_score: 0.5952
linear_accuracy_score: 0.5480464625131996


In [24]:
print(f'ensemble_precision_score: {precision_score(y_test, ensemble_preds)}')
print(f'ensemble_recall_score: {recall_score(y_test, ensemble_preds)}')
print(f'ensemble_accuracy_score: {accuracy_score(y_test, ensemble_preds)}')

ensemble_precision_score: 0.6835664335664335
ensemble_recall_score: 0.6256
ensemble_accuracy_score: 0.5617740232312566
